In [0]:
import uuid
from pyspark.sql import functions as F

class ControlLog:
    def __init__(self, spark, tabela):
        self.spark = spark
        self.tabela = tabela

    def get_last_watermark(self, collection):
        if not self.spark.catalog.tableExists(self.tabela):
            return None
        linha = (self.spark.table(self.tabela)
                    .filter((F.col("collection") == collection) & (F.col("status") == "SUCCESS"))
                    .orderBy(F.col("end_time").desc())
                    .limit(1)
                    .collect())
        return linha[0]["watermark_final"] if linha else None

    def registrar(self, collection, load_type, watermark_inicial, watermark_final,
                  qtd_lida, qtd_gravada, start_time, end_time, status, mensagem_erro=None):
        duracao = (end_time - start_time).total_seconds()

        df = self.spark.range(1).select(
            F.lit(str(uuid.uuid4())).cast("string").alias("_ingestion_id"),
            F.lit(collection).cast("string").alias("collection"),
            F.lit(load_type).cast("string").alias("load_type"),
            F.lit(str(watermark_inicial) if watermark_inicial else None).cast("string").alias("watermark_inicial"),
            F.lit(str(watermark_final) if watermark_final else None).cast("string").alias("watermark_final"),
            F.lit(qtd_lida).cast("int").alias("qtd_lida_origem"),
            F.lit(qtd_gravada).cast("int").alias("qtd_gravada_destino"),
            F.lit(start_time).cast("timestamp").alias("start_time"),
            F.lit(end_time).cast("timestamp").alias("end_time"),
            F.lit(duracao).cast("double").alias("duracao_seg"),
            F.lit(status).cast("string").alias("status"),
            F.lit(mensagem_erro).cast("string").alias("mensagem_erro"),
        )

        df.write.format("delta").mode("append").saveAsTable(self.tabela)